# 📊 Visualización Interactiva — Plotly Express
### Taller de Programación

---

**Plotly Express** es la API de alto nivel de Plotly. Genera gráficos **interactivos** con una sola línea de código.

| Característica | Matplotlib/Seaborn | Plotly Express |
|---|---|---|
| Interactividad | ❌ Estático | ✅ Zoom, hover, filtros |
| Código necesario | Medio/alto | Muy bajo |
| Exportación | PNG, PDF | HTML, PNG, JSON |
| Uso en producción | Reportes estáticos | Dashboards, apps web |

```bash
pip install plotly
```

### Dataset
El **mismo dataset sintético** de ventas de la notebook anterior:
3 regiones × 4 productos × 12 meses — así pueden comparar directamente el código entre las dos notebooks.

---
## 0. Setup y dataset

In [1]:
# !pip install plotly

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

In [3]:
# ── Dataset sintético (idéntico a la notebook de matplotlib) ──────


MESES    = ['Ene','Feb','Mar','Abr','May','Jun',
             'Jul','Ago','Sep','Oct','Nov','Dic']
REGIONES  = ['Norte', 'Centro', 'Sur']
PRODUCTOS = ['Laptop', 'Monitor', 'Teclado', 'Mouse']
PRECIO    = {'Laptop': 1200, 'Monitor': 400, 'Teclado': 80, 'Mouse': 35}

filas = []
for mes_idx, mes in enumerate(MESES):
    for region in REGIONES:
        for producto in PRODUCTOS:
            base       = {'Laptop': 30, 'Monitor': 50, 'Teclado': 120, 'Mouse': 200}[producto]
            tendencia  = 1 + mes_idx * 0.03
            estacional = 1 + 0.25 * np.sin(2 * np.pi * mes_idx / 12 - np.pi / 2)
            region_factor = {'Norte': 1.2, 'Centro': 1.0, 'Sur': 0.8}[region]
            cantidad   = int(base * tendencia * estacional * region_factor
                             * np.random.uniform(0.85, 1.15))
            filas.append({
                'Mes'      : mes,
                'MesNum'   : mes_idx + 1,
                'Region'   : region,
                'Producto' : producto,
                'Cantidad' : cantidad,
                'Precio'   : PRECIO[producto],
                'Revenue'  : cantidad * PRECIO[producto],
            })

df = pd.DataFrame(filas)



In [4]:
print(f"Filas: {len(df)}")
print(f"Revenue total: ${df['Revenue'].sum():,.0f}")
df.head()

Filas: 144
Revenue total: $2,996,490


,Mes,MesNum,Region,Producto,Cantidad,Precio,Revenue
0,Ene,1,Norte,Laptop,25,1200,30000
1,Ene,1,Norte,Monitor,51,400,20400
2,Ene,1,Norte,Teclado,115,80,9200
3,Ene,1,Norte,Mouse,185,35,6475
4,Ene,1,Centro,Laptop,20,1200,24000


---
## 1. Line Plot — Evolución temporal

> 💡 Pasá el mouse sobre las líneas para ver los valores exactos. Hacé click en la leyenda para mostrar/ocultar series.

In [5]:
# Revenue mensual total
rev_mes = df.groupby('MesNum')['Revenue'].sum().reset_index()
rev_mes['Mes'] = MESES

fig = px.line(
    rev_mes,
    x='Mes', y='Revenue',
    title='Revenue mensual — 2026',
    markers=True,
    labels={'Revenue': 'Revenue ($)', 'Mes': 'Mes'},
)
fig.update_traces(line_width=2.5)
fig.show()

In [6]:
# Múltiples líneas: color=columna → una línea por valor único
rev_mes_region = (
    df.groupby(['MesNum', 'Mes', 'Region'])['Revenue']
    .sum()
    .reset_index()
    .sort_values('MesNum')
)

fig = px.line(
    rev_mes_region,
    x='Mes', y='Revenue', color='Region',
    title='Revenue mensual por región — 2026',
    markers=True,
    labels={'Revenue': 'Revenue ($)'},
)
fig.show()

---
## 2. Bar Plot — Comparación entre categorías

In [7]:
# Revenue por producto
rev_prod = (
    df.groupby('Producto')['Revenue']
    .sum()
    .reset_index()
    .sort_values('Revenue', ascending=False)
)

fig = px.bar(
    rev_prod,
    x='Producto', y='Revenue',
    color='Producto',
    title='Revenue total por producto — 2026',
    text_auto='.3s',           # etiquetas automáticas
    labels={'Revenue': 'Revenue ($)'},
)
fig.update_traces(textposition='outside')
fig.show()

In [8]:
# Barras agrupadas: producto × región
rev_prod_reg = (
    df.groupby(['Producto', 'Region'])['Revenue']
    .sum()
    .reset_index()
)

fig = px.bar(
    rev_prod_reg,
    x='Producto', y='Revenue', color='Region',
    barmode='group',           # 'group' | 'stack' | 'overlay'
    title='Revenue por producto y región — 2026',
    labels={'Revenue': 'Revenue ($)'},
)
fig.show()

In [9]:
# Barras apiladas al 100% — muestra proporciones
fig = px.bar(
    rev_prod_reg,
    x='Producto', y='Revenue', color='Region',
    barmode='stack',
    title='Participación de cada región por producto',
    labels={'Revenue': 'Revenue ($)'},
)
fig.show()

---
## 3. Histogram — Distribución

In [10]:
fig = px.histogram(
    df,
    x='Revenue',
    nbins=30,
    title='Distribución de Revenue por transacción',
    labels={'Revenue': 'Revenue ($)', 'count': 'Frecuencia'},
    marginal='box',            # agrega un box plot en el margen superior
)
fig.show()

In [11]:
# Histograma con curva KDE superpuesta
fig = px.histogram(
    df,
    x='Revenue', color='Producto',
    nbins=20,
    barmode='overlay',
    opacity=0.65,
    title='Distribución de Revenue por producto',
    labels={'Revenue': 'Revenue ($)'},
    marginal='violin',
)
fig.show()

---
## 4. Box Plot y Violin Plot

In [12]:
fig = px.box(
    df,
    x='Producto', y='Revenue', color='Region',
    title='Distribución de Revenue — producto × región',
    labels={'Revenue': 'Revenue ($)'},
    points='outliers',         # 'all' | 'outliers' | False
)
fig.show()

In [13]:
fig = px.violin(
    df,
    x='Producto', y='Revenue', color='Producto',
    box=True,                  # incluye box plot interno
    points='all',              # muestra todos los puntos
    title='Violin plot de Revenue por producto',
    labels={'Revenue': 'Revenue ($)'},
)
fig.show()

---
## 5. Scatter Plot — Relación entre variables

In [14]:
fig = px.scatter(
    df,
    x='Cantidad', y='Revenue',
    color='Region',
    symbol='Producto',         # forma diferente por producto
    hover_data=['Mes'],        # dato extra al pasar el mouse
    title='Cantidad vs Revenue — por región y producto',
    labels={'Revenue': 'Revenue ($)', 'Cantidad': 'Unidades vendidas'},
    trendline='ols',           # línea de regresión
    trendline_scope='overall', # una sola línea para todos los grupos
)
fig.show()

---
## 6. Heatmap

In [15]:
# Revenue por mes y producto
pivot = (
    df.groupby(['Mes', 'Producto'])['Revenue']
    .sum()
    .unstack()
    .reindex(MESES)
)

fig = px.imshow(
    pivot,
    color_continuous_scale='YlOrRd',
    title='Revenue mensual por producto — 2026',
    labels={'x': 'Producto', 'y': 'Mes', 'color': 'Revenue ($)'},
    text_auto='.3s',
    aspect='auto',
)
fig.show()

---
## 7. Treemap — Composición jerárquica

Gráfico propio de Plotly, muy útil para mostrar **proporciones anidadas**. Aquí: qué parte del revenue total aporta cada región y dentro de ella cada producto.

In [16]:
rev_jerarquico = df.groupby(['Region', 'Producto'])['Revenue'].sum().reset_index()

fig = px.treemap(
    rev_jerarquico,
    path=['Region', 'Producto'],   # jerarquía: primero región, luego producto
    values='Revenue',
    color='Revenue',
    color_continuous_scale='Blues',
    title='Composición del Revenue — Región → Producto',
)
fig.update_traces(textinfo='label+percent parent')
fig.show()

---
## 8. Sunburst — Composición circular

In [17]:
fig = px.sunburst(
    rev_jerarquico,
    path=['Region', 'Producto'],
    values='Revenue',
    color='Revenue',
    color_continuous_scale='RdBu',
    title='Composición del Revenue (Sunburst)',
)
fig.show()

---
## 9. Subplots interactivos con `make_subplots`

In [18]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Revenue mensual',
        'Revenue por producto',
        'Distribución por región',
        'Cantidad por producto',
    ]
)

# Panel 1 — line
rev_mes = df.groupby('MesNum')['Revenue'].sum().reset_index()
fig.add_trace(
    go.Scatter(x=MESES, y=rev_mes['Revenue'], mode='lines+markers',
               name='Revenue', line_color='steelblue'),
    row=1, col=1
)

# Panel 2 — bar
rev_prod = df.groupby('Producto')['Revenue'].sum().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=rev_prod.index, y=rev_prod.values, name='Producto',
           marker_color=px.colors.qualitative.T10[:4]),
    row=1, col=2
)

# Panel 3 — box
for region in REGIONES:
    datos = df[df['Region'] == region]['Revenue']
    fig.add_trace(
        go.Box(y=datos, name=region, showlegend=False),
        row=2, col=1
    )

# Panel 4 — barh
cant_prod = df.groupby('Producto')['Cantidad'].sum().sort_values()
fig.add_trace(
    go.Bar(x=cant_prod.values, y=cant_prod.index, orientation='h',
           name='Cantidad', showlegend=False,
           marker_color=px.colors.sequential.Blues[2:]),
    row=2, col=2
)

fig.update_layout(
    title_text='Dashboard de ventas — 2026',
    title_font_size=16,
    height=600,
    showlegend=False,
)
fig.show()

---
## 10. Exportar gráficos

Plotly puede exportar como HTML (interactivo) o como imagen estática.

In [19]:
# ── Exportar como HTML interactivo ───────────────────────────────
# El archivo se puede abrir en cualquier browser sin instalar nada.

rev_mes_region = (
    df.groupby(['MesNum', 'Mes', 'Region'])['Revenue']
    .sum().reset_index().sort_values('MesNum')
)
fig = px.line(
    rev_mes_region,
    x='Mes', y='Revenue', color='Region',
    title='Revenue mensual por región — 2026',
    markers=True,
)

fig.write_html('revenue_por_region.html')
print("✅ Guardado: revenue_por_region.html")

# ── Exportar como imagen (requiere: pip install kaleido) ─────────
# fig.write_image('revenue_por_region.png')

✅ Guardado: revenue_por_region.html


---
## ✏️ Ejercicios

**Ejercicio 1.** Hacé un line plot de la **cantidad vendida** mensual por producto. Usá `markers=True` y `hover_data` para mostrar el revenue al pasar el mouse.

**Ejercicio 2.** Creá un bar chart apilado al 100% (`barmode='stack'`) que muestre la participación de cada producto dentro del revenue mensual.

**Ejercicio 3.** Hacé un scatter plot de `Cantidad` vs `Revenue` usando `size='Precio'` para que el tamaño del punto represente el precio unitario. ¿Qué observás?

**Ejercicio 4.** Creá un treemap con la jerarquía `Producto → Región` (al revés que el ejemplo). ¿Cambia la lectura del gráfico?

**Ejercicio 5.** Exportá el dashboard de subplots (sección 9) como HTML y abrilo en el browser.

In [20]:
# Ejercicio 1


In [21]:
# Ejercicio 2


In [22]:
# Ejercicio 3


In [23]:
# Ejercicio 4


In [24]:
# Ejercicio 5
